In [ ]:
# adult_income_fairness.ipynb

# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from fairlearn.metrics import MetricFrame, selection_rate
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

# Load dataset (UCI Adult Income)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country",
    "income"
]
df = pd.read_csv(url, header=None, names=columns, na_values=" ?", skipinitialspace=True)

# Drop rows with missing values
df.dropna(inplace=True)

# Binary target: >50K = 1, <=50K = 0
df['income'] = df['income'].map({">50K": 1, "<=50K": 0})

# Features and target
X = df.drop("income", axis=1)
y = df["income"]

# Sensitive features: sex and race
sensitive_sex = df['sex']
sensitive_race = df['race']

# Train-test split
X_train, X_test, y_train, y_test, s_sex_train, s_sex_test, s_race_train, s_race_test = train_test_split(
    X, y, sensitive_sex, sensitive_race, test_size=0.3, random_state=42
)

# Preprocessing
categorical = X.select_dtypes(include=["object"]).columns
numeric = X.select_dtypes(exclude=["object"]).columns

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", StandardScaler(), numeric)
])

# Pipeline
clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

# Train
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# --- Fairness evaluation by sex ---
metric_frame_sex = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=s_sex_test
)

print("\nFairness by sex:\n", metric_frame_sex.by_group)
metric_frame_sex.by_group.plot(kind="bar", figsize=(8,5))
plt.title("Fairness metrics by sex")
plt.show()

# --- Fairness evaluation by race ---
metric_frame_race = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=s_race_test
)

print("\nFairness by race:\n", metric_frame_race.by_group)
metric_frame_race.by_group.plot(kind="bar", figsize=(10,5))
plt.title("Fairness metrics by race")
plt.show()

# --- Mitigation using ExponentiatedGradient (Demographic Parity, sex as example) ---
constraint = DemographicParity()
mitigator = ExponentiatedGradient(clf, constraints=constraint)
mitigator.fit(X_train, y_train, sensitive_features=s_sex_train)

y_pred_mitigated = mitigator.predict(X_test)

mf_mitigated = MetricFrame(
    metrics={"accuracy": accuracy_score, "selection_rate": selection_rate},
    y_true=y_test,
    y_pred=y_pred_mitigated,
    sensitive_features=s_sex_test
)

print("\nMitigated metrics by sex:\n", mf_mitigated.by_group)
